# Entity Sanity Checks

Validate ontology entity outputs and export all summary tables to CSV.


In [ ]:
from pathlib import Path
import pandas as pd


In [ ]:
BASE = Path('../../exploration/tmp/ontology')
EXPORT_DIR = Path('../../exploration/tmp/notebook_exports/02_entity_sanity_checks')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'geographic': BASE / 'geographic.csv',
    'economic': BASE / 'economic.csv',
    'apartment_market': BASE / 'apartment_market.csv',
}

missing = [name for name, path in PATHS.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing entity files: {missing}. Run build_entities first.')

df_geo = pd.read_csv(PATHS['geographic'])
df_econ = pd.read_csv(PATHS['economic'])
df_apt = pd.read_csv(PATHS['apartment_market'])

dfs = {'geographic': df_geo, 'economic': df_econ, 'apartment_market': df_apt}
print('Export dir:', EXPORT_DIR.resolve())


In [ ]:
table_summary = pd.DataFrame([{
    'entity_table': name,
    'rows': len(df),
    'cols': len(df.columns),
    'columns': ', '.join(df.columns),
} for name, df in dfs.items()])
table_summary.to_csv(EXPORT_DIR / 'table_summary.csv', index=False)
table_summary


In [ ]:
null_rows = []
for table_name, df in dfs.items():
    for col in df.columns:
        null_rows.append({
            'entity_table': table_name,
            'column': col,
            'null_pct': round(df[col].isna().mean() * 100, 2),
            'n_null': int(df[col].isna().sum()),
        })
null_df = pd.DataFrame(null_rows).sort_values(['null_pct', 'entity_table', 'column'], ascending=[False, True, True])
null_df.to_csv(EXPORT_DIR / 'null_rates.csv', index=False)
null_df


In [ ]:
key_checks = []
for table_name, df in dfs.items():
    key_checks.append({
        'entity_table': table_name,
        'check_type': 'entity_id_uniqueness',
        'duplicate_count': int(df['entity_id'].duplicated().sum()),
        'unique_count': int(df['entity_id'].nunique()),
        'row_count': int(len(df)),
    })

if {'geography_entity_id', 'period'}.issubset(df_econ.columns):
    key_checks.append({
        'entity_table': 'economic',
        'check_type': 'geography_period_uniqueness',
        'duplicate_count': int(df_econ.duplicated(subset=['geography_entity_id', 'period']).sum()),
        'unique_count': int(df_econ[['geography_entity_id', 'period']].drop_duplicates().shape[0]),
        'row_count': int(len(df_econ)),
    })

if {'geography_entity_id', 'period'}.issubset(df_apt.columns):
    key_checks.append({
        'entity_table': 'apartment_market',
        'check_type': 'geography_period_uniqueness',
        'duplicate_count': int(df_apt.duplicated(subset=['geography_entity_id', 'period']).sum()),
        'unique_count': int(df_apt[['geography_entity_id', 'period']].drop_duplicates().shape[0]),
        'row_count': int(len(df_apt)),
    })

key_checks_df = pd.DataFrame(key_checks)
key_checks_df.to_csv(EXPORT_DIR / 'key_checks.csv', index=False)
key_checks_df


In [ ]:
period_cov = []
for table_name, df in [('economic', df_econ), ('apartment_market', df_apt)]:
    tmp = df.copy()
    tmp['period_dt'] = pd.to_datetime(tmp['period'], errors='coerce')
    period_cov.append({
        'entity_table': table_name,
        'min_period': tmp['period_dt'].min(),
        'max_period': tmp['period_dt'].max(),
        'n_unique_periods': int(tmp['period_dt'].nunique()),
        'n_invalid_periods': int(tmp['period_dt'].isna().sum()),
    })
period_cov_df = pd.DataFrame(period_cov)
period_cov_df.to_csv(EXPORT_DIR / 'period_coverage.csv', index=False)
period_cov_df


In [ ]:
geo_ids = set(df_geo['entity_id'].dropna().astype(str))
econ_missing_geo = df_econ[~df_econ['geography_entity_id'].astype(str).isin(geo_ids)]
apt_missing_geo = df_apt[~df_apt['geography_entity_id'].astype(str).isin(geo_ids)]

ref_integrity_df = pd.DataFrame([
    {'entity_table': 'economic', 'rows_missing_geography_key': int(len(econ_missing_geo))},
    {'entity_table': 'apartment_market', 'rows_missing_geography_key': int(len(apt_missing_geo))},
])
ref_integrity_df.to_csv(EXPORT_DIR / 'referential_integrity.csv', index=False)
ref_integrity_df


In [ ]:
econ_desc = df_econ['unemployment_rate'].describe().to_frame(name='unemployment_rate')
econ_desc.to_csv(EXPORT_DIR / 'economic_unemployment_describe.csv')
econ_desc


In [ ]:
apt_desc = df_apt[['rent_index', 'rent_growth_1m', 'rent_growth_12m']].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
apt_desc.to_csv(EXPORT_DIR / 'apartment_market_describe.csv')
apt_desc
